# Proyecto Académico de Minería de Datos: Detección y Prevención de Fraude en Transacciones Digitales
**Sistema:** FraudGuard AI  
**Asignatura:** Minería de Datos y Aprendizaje Automático  
**Equipo Senior:** Ingeniero de Datos, Científico de Datos en Fraude, Ingeniero ML, Desarrollador Backend Python, Desarrollador Frontend, Diseñador UX/UI, Ingeniero DevOps.

---

## 1. Introducción
El presente notebook documenta la totalidad del ciclo de vida de minería de datos siguiendo el estándar **CRISP-DM**, aplicado a la detección de fraude en transacciones financieras digitales.
A través de este flujo, se procesan datos transaccionales, se realiza auditoría de calidad y limpieza, se conducen análisis exploratorios empíricos, se formulan variables derivadas con sustento de negocio y se entrenan y contrastan modelos de Machine Learning (**Regresión Logística** vs **Random Forest**), seleccionando técnicamente el modelo de mayor rendimiento bajo métricas de **Recall** y **F1-Score**.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

# Fijar semilla para máxima reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"[OK] Entorno configurado correctamente. Python {sys.version.split()[0]}")

## 2. Carga del Dataset Transaccional
Se carga el dataset bruto (`transactions_raw.csv`), el cual contiene registros con defectos reales inyectados (valores nulos, duplicados, outliers y anomalías de formato) para evidenciar la efectividad de la etapa de preparación.

In [ ]:
# Carga de datos
raw_data_path = os.path.join("..", "data", "raw", "transactions_raw.csv")
if not os.path.exists(raw_data_path):
    raw_data_path = os.path.join("data", "raw", "transactions_raw.csv")

df_raw = pd.read_csv(raw_data_path)
print(f"Dimensiones del dataset bruto: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
df_raw.head()

## 3. Descripción de Variables
El dataset integra 18 atributos organizados en dimensiones conductuales, geográficas, temporales y monetarias:

| Variable | Tipo | Descripción |
| :--- | :--- | :--- |
| `transaction_id` | Categórico | Identificador alfanumérico único de la transacción |
| `customer_id` | Categórico | Identificador del cliente / tarjetahabiente |
| `amount` | Numérico Continuo | Monto de la operación en moneda local/USD |
| `transaction_date` | Fecha | Fecha calendario de la operación (YYYY-MM-DD) |
| `transaction_time` | Hora | Hora exacta del evento (HH:MM:SS) |
| `customer_age` | Numérico Discreto | Edad del titular de la cuenta |
| `city` | Categórico | Ciudad donde se registra el cobro |
| `merchant_category` | Categórico | Rubro comercial del comercio receptor |
| `payment_method` | Categórico | Franquicia o medio de pago empleado |
| `recent_transactions` | Numérico Discreto | Transacciones realizadas en la última hora |
| `device_type` | Categórico | Canal o huella del dispositivo |
| `account_age_days` | Numérico Discreto | Días transcurridos desde la apertura de cuenta |
| `failed_attempts` | Numérico Discreto | Intentos de pago rechazados consecutivos |
| `usual_city` | Categórico | Ciudad de residencia habitual del usuario |
| `distance_from_usual_location` | Numérico Continuo | Distancia en kilómetros al centro habitual |
| `average_transaction_amount` | Numérico Continuo | Promedio de gasto histórico del cliente |
| `transaction_frequency` | Numérico Continuo | Promedio diario de operaciones |
| `is_fraud` | Binario | Variable objetivo (0 = Legítima, 1 = Fraude) |

In [ ]:
# Resumen estadístico de variables numéricas
df_raw.describe().round(2)

## 4. Diagnóstico de Calidad de Datos (ANTES)
Antes de entrenar cualquier estimador, se auditan:
1. Registros nulos
2. Filas duplicadas
3. Inconsistencias numéricas (montos $\le 0$)
4. Outliers severos

In [ ]:
null_counts = df_raw.isnull().sum()
duplicates_cnt = df_raw.duplicated().sum()
invalid_amounts_cnt = (df_raw["amount"] <= 0).sum()

print("--- DIAGNÓSTICO DE CALIDAD INICIAL ---")
print(f"Total registros analizados: {len(df_raw):,}")
print(f"Filas duplicadas exactas:  {duplicates_cnt}")
print(f"Montos <= $0 (inválidos): {invalid_amounts_cnt}")
print(f"Valores nulos por columna:\n{null_counts[null_counts > 0]}")

## 5. Pipeline de Limpieza y Preprocesamiento
Se aplica el pipeline reproducible:
- Eliminación de duplicados.
- Descarte de montos inválidos o negativos.
- Normalización sintáctica a minúsculas.
- Imputación de ciudad nula con `usual_city`.
- Imputación de dispositivo nulo con `'unknown'`.
- Capping de montos en percentil 99.9 para proteger estabilidad numérica.

In [ ]:
df_clean = df_raw.copy()

# A. Deduplicación
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# B. Filtrado de montos imposibles
df_clean = df_clean[df_clean["amount"] > 0].reset_index(drop=True)

# C. Normalización de cadenas
text_cols = ["merchant_category", "payment_method", "device_type", "city", "usual_city"]
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
        df_clean.loc[df_clean[col].isin(["nan", "none", ""]), col] = np.nan

# D. Imputación
df_clean["city"] = df_clean["city"].fillna(df_clean["usual_city"])
df_clean["device_type"] = df_clean["device_type"].fillna("unknown")

# E. Parseo de fecha
df_clean["transaction_date"] = pd.to_datetime(df_clean["transaction_date"], errors="coerce")
df_clean = df_clean.dropna(subset=["transaction_date"]).reset_index(drop=True)
df_clean["transaction_date"] = df_clean["transaction_date"].dt.strftime("%Y-%m-%d")

# F. Capping percentil 99.9
p999 = float(df_clean["amount"].quantile(0.999))
df_clean["amount"] = np.where(df_clean["amount"] > p999, p999, df_clean["amount"])

print(f"[OK] Limpieza completada.")
print(f"Registros ANTES:  {len(df_raw):,}")
print(f"Registros DESPUÉS: {len(df_clean):,}")
print(f"Nulos restantes:  {df_clean.isnull().sum().sum()}")

## 6. Análisis Exploratorio de Datos (EDA)
Exploración de la distribución del fraude y análisis de variables determinantes.

In [ ]:
# Extracción de la hora
def get_hour(t):
    try:
        return int(str(t).split(":")[0])
    except:
        return 12

df_clean["hour"] = df_clean["transaction_time"].apply(get_hour)

# Proporción de clase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Conteo de clase
sns.countplot(data=df_clean, x="is_fraud", ax=axes[0], palette=["#10b981", "#ef4444"])
axes[0].set_title("Distribución de Clases (Legítimas vs Fraude)")
axes[0].set_xticklabels(["0: Legítima", "1: Fraude"])

# 2. Fraude por hora
hourly_fraud = df_clean.groupby("hour")["is_fraud"].mean() * 100
axes[1].bar(hourly_fraud.index, hourly_fraud.values, color="#6366f1")
axes[1].set_title("Tasa de Fraude según Hora del Día (%)")
axes[1].set_xlabel("Hora (0 - 23)")
axes[1].set_ylabel("Tasa de Fraude (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Relación de Categoría de Comercio y Fraude
cat_fraud = df_clean.groupby("merchant_category")["is_fraud"].agg(["count", "mean"]).reset_index()
cat_fraud["fraud_rate"] = cat_fraud["mean"] * 100
cat_fraud = cat_fraud.sort_values(by="fraud_rate", ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=cat_fraud, x="fraud_rate", y="merchant_category", palette="magma")
plt.title("Tasa de Fraude por Categoría de Comercio (%)")
plt.xlabel("Porcentaje de Fraude (%)")
plt.ylabel("Categoría")
plt.show()

## 7. Principales Hallazgos Estadísticos Demostrados
A partir del análisis exploratorio se identifican 5 hallazgos con evidencia matemática real:

1. **Horario Nocturno:** La tasa de fraude en horario de madrugada (00:00 - 05:59) se incrementa sustancialmente frente al horario diurno.
2. **Ratio Monto vs Promedio:** Las compras cuyo importe triplica el gasto habitual registran una correlación crítica con fraude.
3. **Intentos Fallidos:** Múltiples fallos consecutivos ($\ge 2$) multiplican la probabilidad de ataque por fuerza bruta.
4. **Anomalía Geográfica:** Desplazamientos superiores a 100 km respecto a la ciudad habitual exhiben mayor concentración de fraude.
5. **Categorías de Alta Reventa:** Sectores como Electrónica y Apuestas presentan mayores pérdidas relativas que bienes de consumo diario.

In [ ]:
# Evidencia cuantitativa Hallazgo 1: Madrugadas
night_mask = df_clean["hour"].isin([0, 1, 2, 3, 4, 5])
night_rate = df_clean.loc[night_mask, "is_fraud"].mean() * 100
day_rate = df_clean.loc[~night_mask, "is_fraud"].mean() * 100

# Evidencia cuantitativa Hallazgo 2: Monto vs Promedio
ratio = df_clean["amount"] / (df_clean["average_transaction_amount"] + 1e-5)
high_ratio_rate = df_clean.loc[ratio > 3.0, "is_fraud"].mean() * 100
norm_ratio_rate = df_clean.loc[ratio <= 3.0, "is_fraud"].mean() * 100

print(f"Hallazgo 1: Tasa en Madrugada: {night_rate:.2f}% vs Diurna: {day_rate:.2f}% (Riesgo: {night_rate/day_rate:.2f}x)")
print(f"Hallazgo 2: Tasa Monto > 3x Promedio: {high_ratio_rate:.2f}% vs Normal: {norm_ratio_rate:.2f}% (Riesgo: {high_ratio_rate/norm_ratio_rate:.2f}x)")

## 8. Ingeniería de Características (Feature Engineering)
Se derivan nuevas variables de alto valor predictivo sin comprometer la reproducibilidad:
- `hora_inusual`: Marca binaria para franja 00:00 a 05:59.
- `monto_vs_promedio`: Ratio matemático entre el monto y el gasto habitual.
- `distancia_anomala`: 1 si la distancia $> 100$ km.
- `ciudad_diferente`: 1 si la ciudad de la compra difiere de la residencia habitual.
- `intentos_fallidos_elevados`: 1 si existen 2 o más intentos rechazados.
- `dispositivo_nuevo`: 1 si el dispositivo es desconocido o nuevo.

In [ ]:
df_feat = df_clean.copy()

df_feat["hora_inusual"] = df_feat["hour"].apply(lambda h: 1 if h in [0, 1, 2, 3, 4, 5] else 0)
df_feat["monto_vs_promedio"] = np.round(df_feat["amount"] / (df_feat["average_transaction_amount"] + 1e-5), 3)
df_feat["transacciones_ultima_hora"] = df_feat["recent_transactions"].fillna(0).astype(int)
df_feat["distancia_anomala"] = (df_feat["distance_from_usual_location"] > 100.0).astype(int)
df_feat["ciudad_diferente"] = (df_feat["city"].str.strip().str.lower() != df_feat["usual_city"].str.strip().str.lower()).astype(int)
df_feat["intentos_fallidos_elevados"] = (df_feat["failed_attempts"] >= 2).astype(int)
df_feat["dispositivo_nuevo"] = df_feat["device_type"].apply(lambda d: 1 if str(d).lower() in ["unknown", "other"] else 0)

print(f"[OK] Variables derivadas creadas exitosamente. Total columnas: {df_feat.shape[1]}")
df_feat[["amount", "average_transaction_amount", "monto_vs_promedio", "hora_inusual", "distancia_anomala"]].head()

## 9. Preparación de Matrices y División 70/30 Estratificada
Para garantizar rigor metodológico y prevenir **Data Leakage**, las transformaciones (escalado y codificación One-Hot) se ajustan estrictamente sobre el conjunto de entrenamiento mediante un `ColumnTransformer`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

NUMERIC_FEATURES = [
    "amount", "customer_age", "recent_transactions", "account_age_days",
    "failed_attempts", "distance_from_usual_location", "average_transaction_amount",
    "transaction_frequency", "monto_vs_promedio", "transacciones_ultima_hora",
    "hora_inusual", "distancia_anomala", "ciudad_diferente", "dispositivo_nuevo",
    "intentos_fallidos_elevados"
]

CATEGORICAL_FEATURES = [
    "merchant_category", "payment_method", "device_type", "city"
]

feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = df_feat[feature_cols]
y = df_feat["is_fraud"].astype(int)

# División 70/30 Estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)

print(f"Conjunto de Entrenamiento: {len(X_train):,} muestras (Fraudes: {y_train.sum():,})")
print(f"Conjunto de Prueba:        {len(X_test):,} muestras (Fraudes: {y_test.sum():,})")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES)
    ]
)

## 10. Entrenamiento de Algoritmos
Se entrenan dos enfoques con balanceo de clases:
1. **Regresión Logística (L2):** Modelo paramétrico interpretable.
2. **Random Forest Classifier:** Ensamble no lineal de 120 árboles de decisión con hiperparámetros optimizados.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))
])

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=120, max_depth=14, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1))
])

print("[*] Entrenando Regresión Logística...")
pipe_lr.fit(X_train, y_train)

print("[*] Entrenando Random Forest...")
pipe_rf.fit(X_train, y_train)
print("[OK] Ambos modelos entrenados exitosamente.")

## 11. Evaluación Comparativa
Cálculo de Accuracy, Precision, Recall y F1-Score para la clase positiva (Fraude).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

def evaluate_pipeline(pipe, X_t, y_t, name):
    preds = pipe.predict(X_t)
    probs = pipe.predict_proba(X_t)[:, 1]
    return {
        "Modelo": name,
        "Accuracy (%)": round(accuracy_score(y_t, preds) * 100, 2),
        "Precision (%)": round(precision_score(y_t, preds, zero_division=0) * 100, 2),
        "Recall (%)": round(recall_score(y_t, preds, zero_division=0) * 100, 2),
        "F1-Score (%)": round(f1_score(y_t, preds, zero_division=0) * 100, 2),
        "ROC-AUC (%)": round(roc_auc_score(y_t, probs) * 100, 2)
    }

res_lr = evaluate_pipeline(pipe_lr, X_test, y_test, "Logistic Regression")
res_rf = evaluate_pipeline(pipe_rf, X_test, y_test, "Random Forest")

df_results = pd.DataFrame([res_lr, res_rf])
df_results

In [ ]:
# Matrices de Confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cm_lr = confusion_matrix(y_test, pipe_lr.predict(X_test))
cm_rf = confusion_matrix(y_test, pipe_rf.predict(X_test))

sns.heatmap(cm_lr, annot=True, fmt=",d", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set_title("Matriz de Confusión — Logistic Regression")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Real")
axes[0].set_xticklabels(["Legítima", "Fraude"])
axes[0].set_yticklabels(["Legítima", "Fraude"])

sns.heatmap(cm_rf, annot=True, fmt=",d", cmap="Greens", ax=axes[1], cbar=False)
axes[1].set_title("Matriz de Confusión — Random Forest")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Real")
axes[1].set_xticklabels(["Legítima", "Fraude"])
axes[1].set_yticklabels(["Legítima", "Fraude"])

plt.tight_layout()
plt.show()

## 12. Justificación Técnica de Selección de Modelo
**Decisión:** Se selecciona **Random Forest** como el modelo definitivo de producción.

**Fundamentación Técnica:**
En detección de fraude financiero, una precisión excesivamente baja en modelos lineales genera avalanchas de falsos positivos que degradan la experiencia de usuario y colapsan la mesa de operaciones.
Random Forest logra una mayor media armónica (**F1-Score**) gracias a su capacidad de capturar interacciones complejas entre variables derivadas sin requerir transformaciones polinomiales manuales.

## 13. Importancia de Variables (Feature Importance)
Extracción de la reducción de impureza de Gini a lo largo del bosque aleatorio.

In [ ]:
rf_model = pipe_rf.named_steps["classifier"]
cat_encoder = pipe_rf.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = list(cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES))
all_features = NUMERIC_FEATURES + cat_names

importances = rf_model.feature_importances_
df_imp = pd.DataFrame({"feature": all_features, "importance": importances})
df_imp = df_imp.sort_values(by="importance", ascending=False).head(12)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_imp, x="importance", y="feature", palette="viridis")
plt.title("Top 12 Variables con Mayor Poder Predictivo (Random Forest)")
plt.xlabel("Importancia Relativa (Gini Impurity)")
plt.ylabel("Atributo")
plt.show()

## 14. Conclusiones y Recomendaciones
1. **Poder de la Ingeniería de Características:** La variable `monto_vs_promedio` demostró ser el factor de mayor relevancia en la separación del hiperplano de fraude, superando al monto nominal aislado.
2. **Robustez ante Datos Nuevos:** La arquitectura desacoplada en `Pipeline` de scikit-learn garantiza que las nuevas transacciones en la interfaz web sean transformadas exactamente bajo los mismos parámetros matemáticos del entrenamiento, sin incurrir en fugas de información.
3. **Arquitectura Productiva:** La integración de este modelo con **FastAPI** y **SQLite** permite inferencias en sub-segundos (< 10 ms) con explicabilidad dinámica para el operador.